# LOAD DATA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/MRIarchieve.zip'
extract_to = '/content/dataset'

os.makedirs(extract_to, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Dataset unzipped successfully.")

In [ ]:
!pip install torchvision


# LOAD PRE-TRAINED MODEL

In [ ]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms


In [ ]:
resnet = models.resnet50(pretrained=True)
resnet.eval()
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])


# PRE-PROCESSING OF IMAGES

In [ ]:
# Preprocess for ResNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def normalize_slice(slice_2d):
    s = (slice_2d - slice_2d.min()) / (slice_2d.max() - slice_2d.min() + 1e-8)
    s = (s * 255).astype(np.uint8)
    return s


# DEEP FEATURE EXTRACTION

In [ ]:
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

def extract_deep_features(dataset_split_path):
    features_list = []
    
    for label_idx, class_name in enumerate(classes):
        class_folder = os.path.join(dataset_split_path, class_name)
        if not os.path.exists(class_folder):
            continue

        for fname in sorted(os.listdir(class_folder)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_folder, fname)
                
                # Load image and ensure 3-channel RGB
                img = Image.open(img_path).convert('RGB')
                img_tensor = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    # Output shape: (1, 2048, 1, 1) -> Flatten to (2048,)
                    feats = feature_extractor(img_tensor)
                    feats = feats.view(feats.size(0), -1).cpu().numpy().flatten()

                # Append feature vector with label and filename metadata
                row = list(feats) + [class_name, label_idx, fname]
                features_list.append(row)
                
    return features_list

# Extract for Training and Testing sets
train_dir = os.path.join(extract_to, 'Training')
test_dir = os.path.join(extract_to, 'Testing')

train_data = extract_deep_features(train_dir)
test_data = extract_deep_features(test_dir)


# SAVING FEATURE SET

In [ ]:
feature_cols = [f'feat_{i}' for i in range(2048)]
metadata_cols = ['label_name', 'label_idx', 'filename']
columns = feature_cols + metadata_cols

df_train = pd.DataFrame(train_data, columns=columns)
df_test = pd.DataFrame(test_data, columns=columns)

df_train.to_csv('train_features_multiclass.csv', index=False)
df_test.to_csv('test_features_multiclass.csv', index=False)

print(f"Training features saved: {df_train.shape}")
print(f"Testing features saved: {df_test.shape}")